Intermediate Deep Learning - Deep Reinforcement Learning
# Session 1: Foundations of Deep Reinforcement Learning - TD 2
---



**Key Differences from Q-Learning:**
- Neural network instead of Q-table
- No discretization needed (handles continuous states)
- Better generalization across states

### Task 0 ⚙️: Install packages

**Instructions:**

1. Install Pytorch.

In [ ]:
# import sys
# print(sys.executable)

# !"{sys.executable}" -m pip install torch  #MacOS
# !"{sys.executable}" -m pip install torch --index-url https://download.pytorch.org/whl/cu126 # Linux & Windows


|## Part 1: Build the Neural Network
---

### Task 1.1 📝: Create the DQNetwork class

**Instructions:**
The class definition, parent-class initialization, and complete `forward` method are provided.
Fill in only the input and output dimensions of each `nn.Linear` layer, using `state_size`, `hidden_size`, and `n_actions`. The default hidden-layer size is 128.

**Forward pass (provided):** The state passes through two hidden layers with ReLU activations, then through the output layer to produce one Q-value per action. No code needs to be completed in `forward`.

**Question:** Should we apply an activation function to the output layer?

<details>
<summary>Toolbox 🛠️</summary>

- `nn.Linear(input_size, output_size)`: Creates a fully connected layer.

</details>


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

In [ ]:
class DQNetwork(nn.Module):
    def __init__(self, state_size, n_actions, hidden_size=128):
        super(DQNetwork, self).__init__()

        self.fc1 = nn.Linear(state_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, n_actions)

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        return self.fc3(x)


## Part 2: Create the DQN Agent Class
---

### Task 2.1 📝: Initialize the agent

**Instructions:**
The class definition, parameters, device configuration, and MSE loss function are provided. Complete only these two lines:
1. **Create the Q-network:** Set `self.q_network` to a `DQNetwork` with the appropriate state and action dimensions, and move it to `self.device`.
2. **Create the optimizer:** Set `self.optimizer` to an Adam optimizer for the Q-network parameters, using `self.lr` as the learning rate.

<details>
<summary>Toolbox 🛠️</summary>

- `network.to(device)`: Moves the network to the selected device.
- `network.parameters()`: Returns the network parameters.
- `optim.Adam(params, lr)`: Creates an Adam optimizer.

</details>


In [ ]:
class SimpleDQNAgent:
    def __init__(self, lr=0.0005, gamma=0.99, epsilon=1.0, epsilon_decay=0.995, epsilon_min = 0.01, 
                 state_size=4, n_actions=2):
        """
        DQN agent for CartPole-v1
        """

        # Q-Learning hyperparameters
        self.lr = lr
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min

        # Environment parameters
        self.state_size = state_size
        self.n_actions = n_actions

        # Device configuration
        if torch.cuda.is_available():
            self.device = torch.device("cuda")  # GPU
        elif False and torch.backends.mps.is_available():
            self.device = torch.device("mps")  # Metal Performance Shaders for Mac
        else:
            self.device = torch.device("cpu")
        print(f"Using device: {self.device}")

        # Create Q-network
        self.q_network = DQNetwork(state_size, n_actions).to(self.device)

        # Optimizer
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=self.lr)

        # Loss function (provided)
        self.criterion = nn.MSELoss()


### Task 2.2 📝: Implement action selection

**Instructions:**
The epsilon-greedy condition, random action, state-to-tensor conversion, and final action selection are provided. Complete only `q_values = ...` to obtain the Q-values from the network inside the provided `torch.no_grad()` block.

`unsqueeze(0)` adds a batch dimension: a state of shape `(state_size,)` becomes `(1, state_size)`.

**Question:** Why do we use `torch.no_grad()` when obtaining Q-values to choose an action?


In [ ]:
def choose_action(self, state):
    """Epsilon-greedy action selection"""
    if np.random.random() < self.epsilon:
        return np.random.randint(self.n_actions)  # Random action
    else:
        # Convert state to tensor (provided)
        state_tensor = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)

        # Get Q-values from network
        with torch.no_grad():
            q_values = self.q_network(state_tensor)

        # Select action with highest Q-value (provided)
        return torch.argmax(q_values).item()

SimpleDQNAgent.choose_action = choose_action


### Task 2.3 📝: Create the train_step method

The method signature and input-to-tensor conversions are provided below; no code needs to be completed for this task. Add this method to `SimpleDQNAgent`. The following tasks will extend its body to update the network weights.

The state tensors include a batch dimension, and all tensors are created on `self.device`.

**Question:** Why is `action_tensor` created with `torch.long`? Why is `done_tensor` converted to `torch.float32` rather than kept as a Boolean tensor?


In [ ]:
def train_step(self, state, action, reward, next_state, done):
    """Single training step - update network weights"""

    state_tensor = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
    next_state_tensor = torch.tensor(next_state, dtype=torch.float32, device=self.device).unsqueeze(0)
    action_tensor = torch.tensor(action, dtype=torch.long, device=self.device)
    reward_tensor = torch.tensor(reward, dtype=torch.float32, device=self.device)
    done_tensor = torch.tensor(done, dtype=torch.float32, device=self.device)

    # Get the selected current Q-value and detached bootstrap target.
    current_q = self.q_network(state_tensor).gather(1, action_tensor.reshape(1, 1))
    with torch.no_grad():
        next_q_values = self.q_network(next_state_tensor)
        max_next_q = next_q_values.max(dim=1).values
        target_q = reward_tensor + (1.0 - done_tensor) * self.gamma * max_next_q
    # Compute loss
    loss = self.criterion(current_q.squeeze(-1), target_q)

    # Optimize the network
    self.optimizer.zero_grad()
    loss.backward()
    self.optimizer.step()

    return loss.item()

SimpleDQNAgent.train_step = train_step


### Task 2.4 📝: Calculate current and target Q-values

**Instructions:**
Add the code below to `train_step`, after the tensor conversions from Task 2.3. The current Q-value calculation is provided. Complete only the target Q-value calculation inside `torch.no_grad()`.

The network returns one Q-value per action. `gather` selects the value for the action taken. Since `action_tensor` is a scalar in this exercise, `reshape(1, 1)` gives the index the shape required by `gather`. The resulting `current_q` has shape `(1, 1)`.

<details>
<summary>Hint 💡</summary>

1. Pass `next_state_tensor` through the Q-network.
2. Find the largest next-state Q-value along the action dimension (`dim=1`).
3. Combine the immediate reward with the discounted maximum next-state Q-value. Use `done_tensor` to omit the future-value term for a terminal state.
4. Keep the target computation inside the provided `torch.no_grad()` block.

</details>

<details>
<summary>Toolbox 🛠️</summary>

- `tensor.max(dim=1)` returns both maximum values and their indices; use `.values` to get the values.
- `done_tensor` is 0.0 or 1.0, so `1.0 - done_tensor` can be used as a mask.

</details>

**Question:** Why do we use `torch.no_grad()` when calculating the target Q-value, but not when calculating `current_q`?


The completed target calculation is included in `train_step` above.

```python
# Get the selected current Q-value and detached bootstrap target.
current_q = self.q_network(state_tensor).gather(1, action_tensor.reshape(1, 1))
with torch.no_grad():
    next_q_values = self.q_network(next_state_tensor)
    max_next_q = next_q_values.max(dim=1).values
    target_q = reward_tensor + (1.0 - done_tensor) * self.gamma * max_next_q
```


### Task 2.5 📝: Compute loss and update network

The complete code is provided below; no code needs to be completed. Insert it at the end of `train_step`, after the current and target Q-value calculations from Task 2.4.

`current_q.squeeze(-1)` changes the shape from `(1, 1)` to `(1,)`, matching `target_q`. The method then computes the loss, updates the network weights, and returns the scalar loss.

**Question:** Why do we call `self.optimizer.zero_grad()` before `loss.backward()`? What would happen if we omitted it?


The loss and optimizer update are included in `train_step` above.

```python
# Compute loss
loss = self.criterion(current_q.squeeze(-1), target_q)

# Optimize the network
self.optimizer.zero_grad()
loss.backward()
self.optimizer.step()

return loss.item()
```


### Task 2.6 📝: Add epsilon decay

**Instructions:** 
1. Create a simple `decay_epsilon` method that reduces epsilon by multiplying by decay rate, but never goes below minimum.

In [ ]:
def decay_epsilon(self):
    """Decay exploration rate"""
    self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
SimpleDQNAgent.decay_epsilon = decay_epsilon


## Part 3: Training Loop
---

### Task 3.1 📝: Complete and check the training function

The training function is provided below. Only two changes are required:
1. Complete `agent = ...` by creating a `SimpleDQNAgent`, using the state and action dimensions obtained from the environment.
2. Find and correct one error in the training logic. Explain why the correction is needed.

The rest of the code is provided, including episode loops, loss and reward logging, epsilon decay, and environment cleanup. The function returns the agent, rewards, and losses.


In [ ]:
def train_agent(episodes=1000, render=False):
    """Train the DQN agent"""

    # Create environment
    env = gym.make('CartPole-v1', render_mode='human' if render else None)

    # Get state and action dimensions
    state_size = env.observation_space.shape[0]
    n_actions = env.action_space.n

    print(f"State size: {state_size}, Number of actions: {n_actions}")

    # Create an instance of agent
    agent = SimpleDQNAgent(state_size=state_size, n_actions=n_actions)

    # Initialize empty lists for rewards and losses
    rewards_per_episode = []
    losses_per_episode = []

    print("Starting training...")
    print("=" * 60)

    for episode in range(episodes):

        # Reset environment
        state, _ = env.reset(seed=episode)
        env.action_space.seed(episode)
        total_reward = 0
        episode_losses = []
        terminated = False
        truncated = False

        while not (terminated or truncated):
            # Choose action
            action = agent.choose_action(state)

            # Take action in environment
            next_state, reward, terminated, truncated, _ = env.step(action)

            # Train the agent
            loss = agent.train_step(state, action, reward, next_state, done=terminated)
            episode_losses.append(loss)

            # Move to next state
            state = next_state
            total_reward += reward

        # Decay exploration rate
        agent.decay_epsilon()

        # Store rewards and losses
        rewards_per_episode.append(total_reward)
        avg_loss = np.mean(episode_losses)
        losses_per_episode.append(avg_loss)

        # Print progress every 100 episodes
        if (episode + 1) % 100 == 0:
            avg_reward = np.mean(rewards_per_episode[-100:])
            print(f"Episode {episode + 1}/{episodes}, "
                  f"Avg Reward (last 100): {avg_reward:.2f}, "
                  f"Avg Loss: {avg_loss:.4f},"
                  f"Epsilon: {agent.epsilon:.3f}")

    env.close()
    print("=" * 60)
    print("Training complete!")
    return agent, rewards_per_episode, losses_per_episode


## Part 4: Testing and Visualization
---

### Task 4.1 📝: Test the trained agent

The complete test function is provided below; no code needs to be completed.

**Compared with TD1:**
- TD1 discretizes the state and reads Q-values from a Q-table. Here, the continuous state is converted to a tensor with a batch dimension, moved to `agent.device`, and passed through the Q-network under `torch.no_grad()`.
- Both versions choose the action with the highest Q-value, without exploration or learning during evaluation.
- This version also returns the test rewards and prints their average.


In [ ]:
def test_agent(agent, episodes=5, render=False):
    """Test the trained agent"""
    env = gym.make('CartPole-v1', render_mode='human' if render else None)

    print("\nTesting trained agent...")
    test_rewards = []

    for episode in range(episodes):
        state, _ = env.reset(seed=10000 + episode)
        env.action_space.seed(episode)
        total_reward = 0
        terminated = False
        truncated = False

        while not (terminated or truncated):
            # Choose action
            state_tensor = torch.tensor(
                state, dtype=torch.float32, device=agent.device
                ).unsqueeze(0)
            with torch.no_grad():
                q_values = agent.q_network(state_tensor)
            action = torch.argmax(q_values).item()

            state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward

        test_rewards.append(total_reward)

        print(f"Test Episode {episode + 1}: Reward = {total_reward}")

    env.close()

    print(f"Average Test Reward: {np.mean(test_rewards):.2f}")
    return test_rewards


### Task 4.2 📝: Plot training results

The complete `plot_training_results` function is provided below; no code needs to be completed. Run the cell to define it, then call `plot_training_results(rewards, losses)` after training.

The 2x2 grid shows episode rewards, mean loss per episode, and their moving averages with a window of 100 episodes.


In [ ]:
def plot_training_results(rewards, losses):
    """
    Plot episode rewards, mean losses, and their moving averages

    Args:
        rewards (list): Total reward per episode
        losses (list): Mean training-step loss per episode
    """
    # Create a 2x2 grid of subplots
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    # Plot raw rewards in first subplot
    axes[0, 0].plot(rewards, alpha=0.3)
    axes[0, 0].set_title('Raw Rewards per Episode')
    axes[0, 0].set_xlabel('Episode')
    axes[0, 0].set_ylabel('Total Reward')
    axes[0, 0].grid(True)

    # Calculate and plot moving average in second subplot
    window_size = min(100, len(rewards))
    moving_avg_reward = np.convolve(rewards, np.ones(window_size)/window_size, mode='valid')
    axes[0, 1].plot(moving_avg_reward)
    axes[0, 1].set_title(f'Moving Average Reward (window={window_size})')
    axes[0, 1].set_xlabel('Episode')
    axes[0, 1].set_ylabel('Average Reward')
    axes[0, 1].grid(True)

    # Plot raw losses in third subplot
    axes[1, 0].plot(losses, alpha=0.3)
    axes[1, 0].set_title('Raw losses per Episode')
    axes[1, 0].set_xlabel('Episode')
    axes[1, 0].set_ylabel('Mean Loss per Episode')
    axes[1, 0].grid(True)

    # Plot moving average of losses in fourth subplot
    window_size = min(100, len(rewards))
    moving_avg_losses = np.convolve(losses, np.ones(window_size)/window_size, mode='valid')
    axes[1, 1].plot(moving_avg_losses)
    axes[1, 1].set_title(f'Moving Average Loss (window={window_size})')
    axes[1, 1].set_xlabel('Episode')
    axes[1, 1].set_ylabel('Average Loss')
    axes[1, 1].grid(True)

    # Adjust layout and display
    plt.tight_layout()
    plt.show()


## Part 5: Execution
---

### Task 5.1 🚀: Train and Evaluate the Agent

**Instructions:**
1. Set random seeds for reproducibility (`np.random.seed()`, `torch.manual_seed()`, and `torch.cuda.manual_seed_all()`, `to if using GPU)
2. Run `train_agent` to train the agent
3. Plot training results using `plot_results`
4. Run `test_agent` to evaluate the trained agent
5. Interpret the results

In [ ]:
if __name__ == "__main__":
    # Set random seeds for reproducibility
    seed = 23
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    print("Simple DQN for CartPole-v1")
    print("=" * 60)

    # Train the agent
    agent, rewards, losses = train_agent(episodes=1000, render=False)
    # agent.save_model("dqn_cartpole_model.pth")

    # Print final statistics
    print(f"\nFinal Statistics:")
    print(f"  Average reward (last 100 episodes): {np.mean(rewards[-100:]):.2f}")
    print(f"  Best episode reward: {max(rewards):.2f}")
    print(f"  Final epsilon: {agent.epsilon:.3f}")

    # Plot results
    plot_training_results(rewards, losses)

    # Test the trained agent
    # agent.load_model("dqn_cartpole_model.pth")
    test_agent(agent, episodes=5, render=False)


## Part 6 (Bonus) 🌟: Save and Load Models (Optional)

### Task 6.1: Save and load the agent

The complete `save_model` and `load_model` methods are provided below; no code needs to be completed. Add them to your existing `SimpleDQNAgent` class.

The checkpoint stores the Q-network weights, optimizer state, and epsilon. Loading restores them on `self.device`. Create the receiving agent with the same network dimensions and matching settings.


In [ ]:
def save_model(self, filepath):
    """Save the network, optimizer, and exploration rate."""
    torch.save({
        'model_state_dict': self.q_network.state_dict(),
        'optimizer_state_dict': self.optimizer.state_dict(),
        'epsilon': self.epsilon
    }, filepath)
    print(f"Model saved to {filepath}")

def load_model(self, filepath):
    """Load the network, optimizer, and exploration rate."""
    checkpoint = torch.load(filepath, map_location=self.device)
    self.q_network.load_state_dict(checkpoint['model_state_dict'])
    self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    self.epsilon = checkpoint['epsilon']
    print(f"Model loaded from {filepath}")

SimpleDQNAgent.save_model = save_model
SimpleDQNAgent.load_model = load_model


### Answers
Use a linear output: Q-values are unrestricted real-valued expected returns. `no_grad()` saves memory during action selection and treats the TD target as a constant; the current Q-value needs gradients. Action indices use `long` because `gather` requires integer indices. Float terminal flags allow the arithmetic mask `1.0 - done`. Zeroing gradients prevents unintended accumulation across optimizer steps.

The training-loop bug was wrapping `train_step` in `no_grad()`, which disables the computation graph required by backpropagation. A time-limit truncation still bootstraps; a true terminal state does not. This online network uses correlated samples and a moving target, so learning can oscillate or collapse even if an earlier episode scores well. Evaluate the final greedy policy separately from exploratory training rewards.
